In [ ]:
try:
    import firedrake
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    import firedrake

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from firedrake import *
import matplotlib.pyplot as plt
import numpy as np

from firedrake.petsc import PETSc

In [ ]:
# Get current path: all relative paths that you may use for input/output start from here.
#
# - Colab:  The default path is /content, and your GDrive folder is accessible (if mounted) at /content/drive/MyDrive
#
# - local:  If running on your local machine, current_path is the position WHERE YOU LAUNCHED THE NOTEBOOK KERNEL BY 'jupyter notebook'
#           and NOT the directory in which this ipynb file is saved.
#           If you want to modify your current path, go to the terminal, stop the kernel (ctrl-C + press y), then move to the desired path by
#           cd my/desired/path/starting/from/here
#           and then launch 'jupyter notebook'.
import os
current_path = os.getcwd()
print(current_path)

# my_io_path = '/content/drive/MyDrive/Colab Notebooks/CFD2324/'
my_io_path = current_path+"/"
print(my_io_path)

---
---
# Exercise 1
## Coupling Navier-Stokes with thermal problem, according to Boussinesq buoyancy.

In [ ]:
# Import mesh: set the path correctly!
# See cell before Exercise 1 about current path.
mesh = Mesh('meshes/hotplate.msh')
fig, ax = plt.subplots()
triplot(mesh, axes=ax)
ax.legend(loc='upper left')

In [ ]:
# Function spaces
...

# Data and boundary conditions
nu = Constant(0.1)
k = Constant(1.e-2)
beta = Constant(1.e-3)
g = Constant((0,-9.8))
T_ref = Constant(0)
f = Constant((0.,0.))

...
bcsNS = ... BCs for fluid problem ...

...
bcsT = ... BCs for thermal problem ...

### Variational problems

In [ ]:
def nonlinear_iteration_NS(u, v, p, q, nu, beta, g, T_ref, f, T, u_old):
    # Implementing fixed-point method for Navier-Stokes equations with thermal source.
    # u,p   :   TrialFunctions
    # v,q   :   TestFunctions
    # f     :   rhs of NS momentum equation
    # T     :   temperature Function
    # u_old :   advecting velocity Function

    a = ...
    L = ...

    return a, L

def nonlinear_iteration_thermal(T, eta, k, f, u):
    # Implementing fixed-point method for Navier-Stokes equations with thermal source.
    # T     :   TrialFunction
    # eta   :   TestFunction
    # f     :   rhs of thermal equation
    # u     :   advecting velocity Function

    a = ...
    L = ...

    return a, L

### Initialization and post-processing setup.

In [ ]:
# Initialization
wh = Function(W)
uh, ph = wh.subfunctions
uh.interpolate(u_in)
Th = Function(Z)
Th.interpolate(T_in)

# Plot of initial guess
fig, ax = plt.subplots()
col = tripcolor(ph, axes=ax)
plt.colorbar(col)
plt.title('pressure')
fig, ax = plt.subplots()
col = quiver(uh, axes=ax)
plt.colorbar(col)
plt.title('velocity')
fig, ax = plt.subplots()
col = tripcolor(Th, axes=ax)
plt.colorbar(col)
plt.title('temperature')

# vtk output for Paraview
basename = 'lab11_'
outfileU = File(my_io_path+"output/"+basename+"velocity.pvd")
outfileP = File(my_io_path+"output/"+basename+"pressure.pvd")
outfileT = File(my_io_path+"output/"+basename+"temperature.pvd")
uh.rename("Velocity")   # this name will be used in Paraview
ph.rename("Pressure")   # this name will be used in Paraview
Th.rename("Temperature")   # this name will be used in Paraview
outfileU.write(uh)
outfileP.write(ph)
outfileT.write(Th)

### Definition of the ***linear*** solvers for each nonlinear iteration.

In [ ]:
u, p = TrialFunctions(W)
v, q = TestFunctions(W)
u_old = ...
...
p_old = ...
...
a_NS, L_NS = nonlinear_iteration_NS( ... )
pb_NS = LinearVariationalProblem(a_NS, L_NS, wh, bcsNS)
solver_NS =  LinearVariationalSolver(pb_NS)#, solver_parameters=param)

T = TrialFunction(Z)
eta = TestFunction(Z)
T_old = ...
...
a_T, L_T = nonlinear_iteration_thermal( ... )
pb_T = LinearVariationalProblem(a_T, L_T, Th, bcsT)
solver_T =  LinearVariationalSolver(pb_T)#, solver_parameters=param)

### Iterative algorithm for the solution of the nonlinear problem

In [ ]:
maxit = 100
it = 0
tol = 1e-4
err = tol+1     # >tol in order to enter the loop at the beginning

while it <= maxit and err > tol:
    
    it += 1

    ... solve problems ... # pay attention of extracting the correct quantities to pass from a pb to the other
    
    err = ...
    
    print("Iteration = ", it, " Error = ", err)
    uh.rename("Velocity")
    ph.rename("Pressure")
    Th.rename("Temperature")
    outfileU.write(uh)
    outfileP.write(ph)
    outfileT.write(Th)

    ... update the old solution ...

if it <= maxit:
    print('Nonlinear solver converged in', it, 'iterations.')
else:
    print('Nonlinear solver did NOT converge!\nRelative error =', err, 'after', it, 'iterations.')

In [ ]:
fig, ax = plt.subplots()
col = tripcolor(ph, axes=ax)
plt.colorbar(col)
plt.title('pressure')
fig, ax = plt.subplots()
col = quiver(uh, axes=ax)
plt.colorbar(col)
plt.title('velocity')
fig, ax = plt.subplots()
col = tripcolor(Th, axes=ax)
plt.colorbar(col)
plt.title('temperature')

### Point 4: stabilization

In [ ]:
...